<a href="https://colab.research.google.com/github/Isnor/civilization_sim/blob/feat%2Fadd-notebook/civ_sim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install git+https://github.com/Isnor/civilization_sim.git

# Configure an Experiment

In [ ]:
from civ_sim.simulation.scenario import CivilizationScenario
from civ_sim.simulation.model import CivilizationModel

config = CivilizationScenario()
# tweak the parameters that you want for your experiment; these can be passed into the Scenario directly as kwargs
# or modified with config.<parameter>. Below are the defaults.

# population
# population_initial_size: int = 100
# population_max_size: int = 1000
# population_utility_fn: str = "survival"

# resources
# resources_initial:float = 10.0              # starting resources for each agent
# resources_initial_offspring:float = 5.0     # starting resources for newborns
# resources_survival_cost:float = 1.0         # resources consumed per tick
# resources_rest_gain:float = 0.3             # resources recovered by resting
# resources_forage_gain:tuple[float, float] = (1.5, 3.5)   # [min, max] resources gained by foraging
# resources_reproduction_threshold:float = 15.0  # min resources to reproduce
# resources_reproduction_cost:float = 5.0     # resources spent by each parent
# resources_min_reproduction_age:float = 13   # agents begin to reproduce after this many ticks
# resources_max_reproduction_age:float = 60   # agents stop reproducing after this many ticks
# resources_max_age:float = 80                # agents die after this many ticks

# initial trait distribution; each tuple is (average, std_deviation)
# trait values themselves are clamped to [0, 1]
# traits_curiosity: tuple[float, float] = (.5, .2)
# traits_pattern_recognition: tuple[float, float] = (.5, .2)
# traits_abstraction: tuple[float, float] = (.4, .2)
# traits_memory_narrative: tuple[float, float] = (.5, .2)
# traits_social_desire: tuple[float, float] = (.6, .2)
# traits_dominance: tuple[float, float] = (.5, .2)
# traits_empathy: tuple[float, float] = (.5, .2)
# traits_trust: tuple[float, float] = (.45, .2)
# traits_conformity: tuple[float, float] = (.5, .2)
# traits_risk_tolerance: tuple[float, float] = (.5, .2)
# traits_aggression: tuple[float, float] = (.45, .2)
# traits_industriousness: tuple[float, float] = (.5, .2)
# traits_patience: tuple[float, float] = (.5, .2)
# traits_wonder: tuple[float, float] = (.4, .2)
# traits_attribution_style: tuple[float, float] = (.5, .25)
# traits_reverence: tuple[float, float] = (.4, .2)

# trait drift / growth
# trait_drift_rate: float = .02
# trait_drift_max_deviation: float = .03

# group dynamics
# social_encounter_probability: float = .25
# social_group_formation_threshold: float = .45
# social_cooperation_bonus: float = .30

# heritability
# heritability_variance: float = .10

# inspiration
# inspiration_probability: float = .01

# unknown_player events
# unknown_player_event_probability: float = .01

# endgames
# endgames_max_steps: int = 1000

model = CivilizationModel(scenario=config)
attributor_fraction = 100 * model.attributor_fraction()
modeler_fraction = 100 * model.modeler_fraction()
indifferent_fraction = 100 - attributor_fraction - modeler_fraction

fig, axs = plt.subplots(1, 1, figsize=(4, 4))
axs.pie(
    x=[attributor_fraction, modeler_fraction, indifferent_fraction],
    labels=["attributors", "modelers", "indifferent"],
    autopct='%.0f%%',
)
fig.tight_layout()
fig.show()


In [ ]:
model.run_model() # run until an endgame is reached
# model.run_for() # you can also run it for some duration of time
# model.run_until() # or until some deadline
# model.step() # you can also run a single step of the model if you want to regenerate the charts before the endgame or deadline is reached

# Analyze the Results

Some rudimentary examples for looking at the simulation are provided below.

In [ ]:
import pandas as pd
from pandas import DataFrame
import seaborn as sns
import matplotlib.pyplot as plt

from civ_sim.analysis.chart_helpers import (
    plot_belief_orientations_stacked,
    plot_tech_adoption,
    get_event_tick_series,
    plot_population_with_events,
    plot_trait_boxplot,
    plot_trait_distribution,
    get_event_tick_series,
)


model_df = model.datacollector.get_model_vars_dataframe()
agent_df = model.datacollector.get_agent_vars_dataframe()

 # Get social tech columns and events from model
tech_cols = [c for c in model_df.columns if c.startswith('tech_')]
events = get_event_tick_series(model)

# Create a 2x2 subplot grid
charts, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()

# Row 0: Population trends (left) + Belief orientation trends (right)
plot_population_with_events(axes[0], model_df, events)

plot_belief_orientations_stacked(axes[1], model_df)
# Row 1: Social technology adoption (left) + Event timeline (right)
plot_tech_adoption(axes[2], model_df, tech_cols)

# Event timeline - vertical markers for each event
event_ticks = events['tick'].tolist() if len(events) > 0 else []
colors = ['#e74c3c', '#3498db', '#27ae60', '#e67e22', '#9b59b6']

for i, tick in enumerate(event_ticks):
    color = colors[i % len(colors)]
    # Add event marker
    axes[3].scatter(tick, 0.5, color=color, marker='o', s=60, zorder=3, alpha=0.8)
    # Add vertical line extending up from event
    axes[3].axvline(x=tick, color=color, linestyle=':', alpha=0.5, linewidth=1)

axes[3].set_ylim(-0.1, 1.0)
axes[3].set_yticks([])
axes[3].set_xlabel('Tick')
axes[3].set_title('Events Timeline')
axes[3].grid(True, alpha=0.3)

# Add overall figure title
charts.suptitle('Civilization Simulation Results', fontsize=16, y=1.0)

plt.tight_layout()

In [ ]:
from civ_sim.core.traits import TRAIT_NAMES

fig, axs = plt.subplots(4, 4, figsize=(16, 16))
fig.suptitle('Trait Distributions - All Agents', fontsize=16, y=1.0)

axs = axs.flatten()

for i in range(0, len(TRAIT_NAMES)):
    ax = axs[i]
    plot_trait_distribution(ax=ax, data_agents=agent_df, trait=TRAIT_NAMES[i])

fig.tight_layout()

In [ ]:
from civ_sim.analysis.collectors import _MODEL_TRAIT_REPORTERS

fig, axs = plt.subplots(2, 1, figsize=(16, 16))
axs = axs.flatten()

ax = sns.lineplot(
    data=model_df[[f'avg_{r}' for r in _MODEL_TRAIT_REPORTERS]],
    ax=axs[0],
    linewidth=1.5,
    alpha=0.6
)
ax.set_xlabel('Tick')
ax.set_ylabel('Average Trait Value')
ax.set_title('Trait Evolution Over Time')
ax.legend(loc='best', bbox_to_anchor=(1.05, 1), ncol=2)
ax.grid(True, alpha=0.3)

plot_trait_boxplot(axs[1], agent_df, TRAIT_NAMES)

# these take a lot of CPU/memory
# sns.kdeplot(data_agents, ax=axs[0], x="trust", y="empathy")
# sns.kdeplot(data_agents, ax=axs[0], x="wonder", y="reverence")
fig.tight_layout()